# 과제 10 · 05 · 정비 이력을 시점 경계가 있는 Tool 로 열기

## 현업 시나리오
정비 이력에서 "같은 유형으로 최근에 무엇을 고쳤나"를 찾아야 합니다. 이력 DB 에는 이벤트 **뒤**에 생긴 기록도 있습니다. 그것을 근거로 쓰면 현장에서는 있을 수 없는 근거(미래의 수리 결과)로 초안을 쓰게 되고, 평가 점수도 부풀려집니다.

## 학습 목표
- 함수 시그니처와 docstring 에서 Tool 스키마가 만들어지는 것을 본다(`@tool`).
- `Literal` 인자가 허용값 목록이 되어 함수 앞에서 막히는 것을 확인한다.
- 조회 시점(`before`)으로 이벤트 뒤의 기록을 막고, 시점 형식을 검사한다.
- 읽기 전용 연결이 쓰기를 거부하는 것을 확인한다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · @tool 로 이력 조회를 열기

In [ ]:
# 연습용 정비 이력 DB 를 임시 파일로 만듭니다. 실제 이력(410건)의 모양을 줄인 것입니다.
# 이벤트는 2025-08-22 06:30 에 왔습니다. MR-0103 은 그 **뒤**의 기록입니다.
import sqlite3
import tempfile
from contextlib import closing
from pathlib import Path

PRACTICE_EVENT_AT = "2025-08-22T06:30:00+09:00"
practice_db = Path(tempfile.mkdtemp()) / "practice_history.sqlite"
with closing(sqlite3.connect(practice_db)) as practice_setup:
    practice_setup.execute("CREATE TABLE record (record_id TEXT, occurred_at TEXT, failure_type TEXT, action TEXT)")
    practice_setup.executemany("INSERT INTO record VALUES (?, ?, ?, ?)", [
        ("MR-0101", "2025-07-02T14:00:00+09:00", "HDF", "냉각팬 모듈(PN-CF-3105) 교체"),
        ("MR-0102", "2025-08-10T09:30:00+09:00", "HDF", "흡기 필터(PN-FL-3120) 교체"),
        ("MR-0103", "2025-08-25T11:00:00+09:00", "HDF", "방열판 청소 — 이벤트 뒤의 기록"),
        ("MR-0104", "2025-08-01T08:00:00+09:00", "TWF", "공구(PN-TL-1008) 교체"),
    ])
    practice_setup.commit()

def practice_connect():
    """조회마다 읽기 전용으로 엽니다. mode=ro 면 쓰기 문장은 DB 가 거부합니다."""
    return sqlite3.connect(f"file:{practice_db}?mode=ro", uri=True)

# 스키마를 손으로 적지 않습니다. @tool 이 시그니처와 docstring 에서 만듭니다.
# 관찰 포인트: Literal 로 적은 failure_type 은 스키마에 enum(허용값 목록)으로 나타납니다.
import json
from datetime import datetime
from typing import Literal
from langchain_core.tools import tool

def practice_moment(before):
    try:
        datetime.fromisoformat(before)
    except ValueError as error:
        raise ValueError(f"before 는 ISO 시각이어야 합니다. 받은 값: {before!r}") from error
    return before

@tool
def practice_history_tool(failure_type: Literal["TWF", "HDF", "PWF", "OSF"], before: str) -> str:
    """정비 이력에서 이 유형으로 수리한 최근 기록을 찾는다. before 시각보다 앞선 기록만 돌려준다."""
    with closing(practice_connect()) as db:
        rows = db.execute("SELECT record_id, occurred_at, action FROM record WHERE failure_type = ? "
                          "AND occurred_at < ? ORDER BY occurred_at DESC",
                          (failure_type, practice_moment(before))).fetchall()
    return json.dumps({"records": [dict(zip(("id", "at", "action"), r)) for r in rows]}, ensure_ascii=False)

practice_schema = practice_history_tool.args_schema.model_json_schema()
print("Tool 이름:", practice_history_tool.name)
print("failure_type 허용값:", practice_schema["properties"]["failure_type"]["enum"])
print("설명:", practice_history_tool.description[:40], "…")

**결과 해설** — Tool 이름과 설명, 인자 스키마가 함수에서 그대로 나왔습니다. `failure_type` 은 네 값만 허용하는 enum 입니다. 모델은 이 스키마를 보고 무엇을 어떤 인자로 부를지 정합니다.

### 2단계 · 시점 경계가 미래 기록을 막는다

In [ ]:
# 경계 없이 같은 유형을 모두 찾는 조회와 비교합니다.
# 같은 DB, 같은 유형입니다. 다른 것은 "before 보다 앞선 기록만" 이라는 조건 하나입니다.
# 관찰 포인트: 경계가 없으면 이벤트 3일 뒤의 수리(MR-0103)가 근거로 들어옵니다.
def practice_leaky(failure_type):
    with closing(practice_connect()) as db:
        return [r[0] for r in db.execute(
            "SELECT record_id FROM record WHERE failure_type = ? ORDER BY occurred_at DESC", (failure_type,))]

practice_bounded = [r["id"] for r in json.loads(
    practice_history_tool.invoke({"failure_type": "HDF", "before": PRACTICE_EVENT_AT}))["records"]]
print("경계 있음:", practice_bounded)
print("경계 없음:", practice_leaky("HDF"))

**결과 해설** — 경계가 있으면 이벤트 전의 HDF 수리 두 건(MR-0102, MR-0101)만 나옵니다. 경계가 없으면 이벤트 3일 뒤의 MR-0103 이 맨 앞에 옵니다. 최근 순으로 정렬하므로 **미래 기록이 가장 먼저 근거가 됩니다.** 오류도 경고도 없이 일어납니다.

### 3단계 · 잘못된 인자와 쓰기는 막힌다

In [ ]:
# 세 가지를 시도합니다. 없는 유형, 형식이 틀린 시각, 그리고 읽기 전용 연결에서의 삭제.
# 앞의 둘은 Tool 이 막고, 마지막은 DB 가 막습니다.
# 관찰 포인트: 시각은 문자열로 비교하므로 형식이 틀리면 조용히 틀린 결과를 냅니다. 그래서 형식부터 검사합니다.
from pydantic import ValidationError

practice_blocked = {}
for practice_label, practice_args in (("없는 유형", {"failure_type": "XYZ", "before": PRACTICE_EVENT_AT}),
                                      ("틀린 시각", {"failure_type": "HDF", "before": "8월 22일"})):
    try:
        practice_history_tool.invoke(practice_args)
        practice_blocked[practice_label] = None
    except (ValidationError, ValueError) as error:
        practice_blocked[practice_label] = type(error).__name__
try:
    with closing(practice_connect()) as practice_writer:
        practice_writer.execute("DELETE FROM record")
    practice_blocked["삭제"] = None
except sqlite3.OperationalError as error:
    practice_blocked["삭제"] = str(error)
for practice_label, practice_reason in practice_blocked.items():
    print(f"  {practice_label:6} → {practice_reason}")

assert practice_schema["properties"]["failure_type"]["enum"] == ["TWF", "HDF", "PWF", "OSF"]
assert practice_bounded == ["MR-0102", "MR-0101"], "시점 경계가 지켜지지 않았습니다"
assert "MR-0103" in practice_leaky("HDF"), "경계 없는 조회에 미래 기록이 없습니다(비교가 성립하지 않습니다)"
assert practice_blocked["없는 유형"] == "ValidationError"
assert practice_blocked["틀린 시각"] == "ValueError"
assert practice_blocked["삭제"] and "readonly" in practice_blocked["삭제"]

**결과 해설** — 없는 유형은 함수에 닿기 전에 스키마 검증(`ValidationError`)이 막고, "8월 22일" 같은 시각은 함수 첫 줄의 형식 검사가 막습니다. 삭제는 DB 가 "readonly" 로 거부합니다. Tool 은 근거를 **읽기만** 합니다.

## 직접 해 볼 과제
`practice_try_before` 를 이벤트 시각보다 이른 "2025-08-05T00:00:00+09:00" 이나 더 늦은 "2025-09-01T00:00:00+09:00" 으로 바꿔 보세요. 결과가 어떻게 바뀝니까? 앱이 왜 `before` 에 EventCard 의 `detected_at` 을 그대로 넣으라고 Agent 에게 지시하는지 생각해 보세요.

In [ ]:
# 조회 시점만 바꿉니다. 유형과 DB 는 그대로입니다.
# 늦은 시점을 넣으면 경계가 있어도 미래 기록이 들어옵니다. 경계의 값이 맞아야 경계가 의미 있습니다.
practice_try_before = "2025-08-05T00:00:00+09:00"
print(json.loads(practice_history_tool.invoke({"failure_type": "HDF", "before": practice_try_before}))["records"])

## 중간 결과
Tool 스키마의 enum, 시점 경계가 있을 때와 없을 때의 조회 결과, 그리고 없는 유형·틀린 시각·삭제가 각각 무엇에 막히는지 확인합니다.

## 실패 경계
시점 경계는 `before` 값이 맞을 때만 의미가 있습니다. 모델이 시각을 지어내거나 "오늘"을 넣으면 경계가 있어도 미래 기록이 들어옵니다. 그래서 앱은 Agent 에게 EventCard 의 `detected_at` 을 그대로 넣으라고 지시하고, 결과를 다시 코드로 확인합니다. Test 가 시나리오마다 "돌려준 기록이 모두 이벤트보다 앞선다"를 검사합니다.

## 실제 app 연결
과제 10 App 의 `tools.py::same_type_history` 가 같은 모양의 Tool 이고, `tools.py::_moment` 가 시각 형식을 막습니다. 조회는 `history.py::HistoryStore` 가 Postgres 표에서 맡으며 조회마다 읽기 전용 세션(`postgres.py::connect`)을 새로 엽니다. 이 Notebook 의 임시 SQLite 는 연습용입니다. Tool 은 이것 말고도 판정 기준(`check_criteria`), 과거 오탐 점검, 마지막 공구 교체를 엽니다.

## 다음 Notebook 연결
다음 `06_lookup_agent.ipynb` 에서는 이 Tool 들을 Agent 에게 쥐여 주고, Agent 가 실제로 무엇을 불렀는지 기록에서 확인합니다.